In [ ]:
#notebook adapted from https://github.com/chung-neuroai-lab/SNAP/

import sys

# Add the directory to the Python path
sys.path.append('.')


from brain_score_data import get_neural_data, get_dataloader, LoaderTORCH

In [ ]:
from snap.wrapper import TorchWrapper
from snap.experiment import Experiment
from snap.regression_utils import regression_metric
import snap.models as models
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA



In [ ]:
import os
import numpy as np

# set the data root directory which saves the activations and torch home
data_root = '/path/to/data_root'
os.makedirs(data_root, exist_ok=True)
device = 'cuda'



In [ ]:

regionNames = ["IT","V4"]

activation_pooling = [None,
                      #   'MaxPool_(1,1)',
                      #   'AvgPool_(1,1)',
                      ]

rand_proj_dim = None

pretrained = {True: 'pretrained',
              False: 'untrained'
              }

loader_kwargs = {'batch_size': 800,
                 'shuffle': False,
                 'num_workers': 4,
                 'pin_memory': True,
                 'onehot': True,
                 'labels_from': 'neural_activity'
                }

In [ ]:

modelNames = ["robust_resnet50_l2_3"]


# modelNames = ["resnet152"]
pooling = None
trained = True
n_components = 1000
data_root = f'{data_root}/model_act/'

for region in regionNames:
    data_loader_neural, images, labels = get_neural_data(region=region,
                                            data_root=data_root,
                                            loader_kwargs=loader_kwargs)

    loader_kwargs = {'batch_size': 1024,
                 'shuffle': False,
                 'num_workers': 4,
                 'pin_memory': True,
                 'onehot': True,         }


    imagenet_val = np.load(f"{data_root}/Imagenet_Val_1000_filepaths.npz")
    imagenet_val = imagenet_val["file_paths"]
    val_imagenet_df = pd.DataFrame({"image_names": np.arange(1000), "image_files": imagenet_val})


    imagenet_val_loader, _ = get_dataloader(val_imagenet_df, labels_from = "names", **loader_kwargs)
    imagenet_val_loader= LoaderTORCH(imagenet_val_loader)
    
    for model_name in modelNames:
        explained_var_model = {}
        effective_dimensionality = {}
        data_fname = os.path.join(data_root,
            f"{model_name}_{pretrained[trained]}_{region}.npz")
        os.makedirs(data_root, exist_ok=True)
        print(data_fname)

        model_kwargs = {'name': model_name,
                        'pretrained': trained,
                        'device': device}
        model, layers, identifier = models.get_model(**model_kwargs)

        model_wrapped = TorchWrapper(model,
                                     layers=layers,
                                     identifier=identifier,
                                     activation_pooling=pooling)

        # Create the Experiment Class and pass additional metrics
        regression_kwargs = {'num_trials': 5,
                             'reg': 1e-14,
                             'num_points': 5,
                             }

        metric_fns = [regression_metric]
        exp = Experiment(model_wrapped,
                         metric_fns=metric_fns,
                         rand_proj_dim=rand_proj_dim)

        imagenet_activations = exp.get_activations(imagenet_val_loader())
        pca_transformations = {}
        for layer, act in imagenet_activations.items():
            print("PCA for layer", layer)
            if act.shape[-1] > n_components:
                print("n_initial_features", act.shape[-1])
                pca = PCA(n_components=n_components, random_state=0)
                pca.fit(act)
                explained_var_model[layer] = pca.explained_variance_ratio_
                lambdas = pca.explained_variance_ratio_
                effective_dimensionality[layer] = (np.sum(lambdas)) ** 2 / np.sum(lambdas ** 2)
                print("ed", effective_dimensionality[layer])
            else:
                print("pca not need", act.shape[-1])
                pca = None
            pca_transformations[layer] = pca
        
        activations = exp.get_activations(data_loader_neural())

        for layer, act in activations.items():
            pca = pca_transformations[layer]
            if pca is not None:
                 activations[layer] = pca.transform(act)

        np.savez(data_fname, activations=activations, explained_var = explained_var_model, ed = effective_dimensionality)
